### **台股選股（六條件）**

In [ ]:
# @title
# ============================================================
# 台股選股（六條件）
# ============================================================
#
# 【選股條件】
#
# 1. 6個月績效 < 50%
# 2. RSI(14) > 50
# 3. ADX(14) > 20
# 4. MA50 > MA200
# 5. 今日成交量 > 30日平均成交量 且 30日平均成交量 > 2,000張
# 6. 股價 < MA20 × 0.99
#
# ============================================================

# ============================================================
# 0. 安裝套件
# ============================================================

!pip -q install yfinance pandas numpy requests openpyxl tqdm


# ============================================================
# 1. 載入套件
# ============================================================

import pandas as pd
import numpy as np
import yfinance as yf
import requests
import warnings
import time
import os
from datetime import datetime

from tqdm.auto import tqdm

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

warnings.filterwarnings("ignore")


# ============================================================
# 2. 選股參數
# ============================================================

# ----------------------------
# RSI
# ----------------------------

RSI_PERIOD = 14
RSI_MIN = 50


# ----------------------------
# ADX
# ----------------------------

ADX_PERIOD = 14
ADX_MIN = 20


# ----------------------------
# 均線
# ----------------------------

MA20_PERIOD = 20
MA50_PERIOD = 50
MA200_PERIOD = 200


# ----------------------------
# 成交量
# ----------------------------

VOLUME_PERIOD = 30

# 2,000 張 = 2,000,000 股
MIN_AVG_VOLUME_SHARES = 2_000_000


# ----------------------------
# 6個月績效
# ----------------------------

# 約126個交易日
SIX_MONTH_DAYS = 126

# 6個月績效上限
MAX_6M_RETURN = 50


# ----------------------------
# MA20 條件
# ----------------------------

# 股價 < MA20 × 0.99
MA20_FACTOR = 0.99


# ----------------------------
# Yahoo Finance 批次
# ----------------------------

BATCH_SIZE = 100

# Yahoo 下載失敗時重試次數
MAX_RETRIES = 3

# 每次重試等待秒數
RETRY_DELAY = 3


# ============================================================
# 3. 開始計時
# ============================================================

program_start = time.time()


# ============================================================
# 4. 主標題
# ============================================================

print()
print("=" * 110)
print("★★★★★ 台股選股（六條件） ★★★★★")
print("=" * 110)

print()
print("選股條件：")
print("① 6個月績效 < 50%")
print("② RSI(14) > 50")
print("③ ADX(14) > 20")
print("④ MA50 > MA200")
print("⑤ 今日成交量 > 30日平均量，且平均量 > 2,000張")
print("⑥ 股價 < MA20 × 0.99")
print("=" * 110)


# ============================================================
# 5. 取得台股股票清單
# ============================================================

print()
print("【1/7】正在取得台股股票清單...")

FINMIND_URL = "https://api.finmindtrade.com/api/v4/data"

params = {
    "dataset": "TaiwanStockInfo"
}

try:
    response = requests.get(
        FINMIND_URL,
        params=params,
        timeout=30
    )
    response.raise_for_status()
    json_data = response.json()

except Exception as e:
    raise RuntimeError(
        f"取得 FinMind 股票清單失敗：{e}"
    )


if "data" not in json_data:
    raise RuntimeError(
        "FinMind 沒有回傳股票資料"
    )


stocks = pd.DataFrame(
    json_data["data"]
)


# ============================================================
# 6. 股票資料整理
# ============================================================

stocks["stock_id"] = (
    stocks["stock_id"]
    .astype(str)
)


# ----------------------------
# 上市 / 上櫃
# ----------------------------

stocks = stocks[
    stocks["type"].isin(
        ["twse", "tpex"]
    )
].copy()


# ----------------------------
# 排除 ETF
# ----------------------------

stocks = stocks[
    stocks["industry_category"]
    .fillna("")
    .str.upper()
    != "ETF"
].copy()


# ----------------------------
# 只保留4碼股票
# ----------------------------

stocks = stocks[
    stocks["stock_id"]
    .str.match(r"^\d{4}$")
].copy()


# ----------------------------
# 去除重複
# ----------------------------

stocks = stocks.drop_duplicates(
    subset=["stock_id"]
)


stocks = stocks.reset_index(
    drop=True
)


print(
    f"上市 + 上櫃普通股票："
    f"{len(stocks):,} 檔"
)


# ============================================================
# 7. 建立 Yahoo Finance 股票代號
# ============================================================

print()
print("正在建立 Yahoo Finance 股票代號...")


stocks["symbol"] = np.where(
    stocks["type"].eq("twse"),
    stocks["stock_id"] + ".TW",
    np.where(
        stocks["type"].eq("tpex"),
        stocks["stock_id"] + ".TWO",
        None
    )
)


stocks = stocks.dropna(
    subset=["symbol"]
)


stocks = stocks.reset_index(
    drop=True
)


# ============================================================
# 8. 股票基本資料
# ============================================================

stock_info = stocks[
    [
        "stock_id",
        "stock_name",
        "type",
        "symbol"
    ]
].copy()


stock_info = stock_info.set_index(
    "symbol"
)


# ============================================================
# 9. RSI 計算
# ============================================================

def calculate_rsi(
    close,
    period=14
):
    close = pd.Series(
        close,
        dtype="float64"
    )

    delta = close.diff()

    gain = delta.clip(
        lower=0
    )

    loss = -delta.clip(
        upper=0
    )

    avg_gain = gain.ewm(
        alpha=1 / period,
        adjust=False,
        min_periods=period
    ).mean()

    avg_loss = loss.ewm(
        alpha=1 / period,
        adjust=False,
        min_periods=period
    ).mean()

    rs = (
        avg_gain /
        avg_loss.replace(
            0,
            np.nan
        )
    )

    rsi = (
        100 -
        100 / (1 + rs)
    )

    # 如果平均跌幅為0，RSI應視為100
    rsi = rsi.where(
        avg_loss != 0,
        100
    )

    return rsi


# ============================================================
# 10. ADX 計算
# ============================================================

def calculate_adx(
    high,
    low,
    close,
    period=14
):

    high = pd.Series(
        high,
        dtype="float64"
    )

    low = pd.Series(
        low,
        dtype="float64"
    )

    close = pd.Series(
        close,
        dtype="float64"
    )


    # --------------------------------------------------------
    # True Range
    # --------------------------------------------------------

    tr1 = high - low

    tr2 = (
        high -
        close.shift(1)
    ).abs()

    tr3 = (
        low -
        close.shift(1)
    ).abs()


    tr = pd.concat(
        [
            tr1,
            tr2,
            tr3
        ],
        axis=1
    ).max(
        axis=1
    )


    # --------------------------------------------------------
    # Directional Movement
    # --------------------------------------------------------

    up_move = high.diff()

    down_move = -low.diff()


    plus_dm = pd.Series(
        np.where(
            (
                (up_move > down_move) &
                (up_move > 0)
            ),
            up_move,
            0
        ),
        index=high.index,
        dtype="float64"
    )


    minus_dm = pd.Series(
        np.where(
            (
                (down_move > up_move) &
                (down_move > 0)
            ),
            down_move,
            0
        ),
        index=high.index,
        dtype="float64"
    )


    # --------------------------------------------------------
    # Wilder smoothing
    # --------------------------------------------------------

    atr = tr.ewm(
        alpha=1 / period,
        adjust=False,
        min_periods=period
    ).mean()


    plus_dm_smoothed = (
        plus_dm
        .ewm(
            alpha=1 / period,
            adjust=False,
            min_periods=period
        )
        .mean()
    )


    minus_dm_smoothed = (
        minus_dm
        .ewm(
            alpha=1 / period,
            adjust=False,
            min_periods=period
        )
        .mean()
    )


    # --------------------------------------------------------
    # DI
    # --------------------------------------------------------

    plus_di = (
        100 *
        plus_dm_smoothed /
        atr.replace(
            0,
            np.nan
        )
    )


    minus_di = (
        100 *
        minus_dm_smoothed /
        atr.replace(
            0,
            np.nan
        )
    )


    # --------------------------------------------------------
    # DX
    # --------------------------------------------------------

    denominator = (
        plus_di +
        minus_di
    )


    dx = (
        100 *
        (
            plus_di -
            minus_di
        ).abs() /
        denominator.replace(
            0,
            np.nan
        )
    )


    # --------------------------------------------------------
    # ADX
    # --------------------------------------------------------

    adx = dx.ewm(
        alpha=1 / period,
        adjust=False,
        min_periods=period
    ).mean()


    return adx


# ============================================================
# 11. Yahoo Finance 批次下載函數
# ============================================================

def download_batch(
    batch_symbols,
    retries=MAX_RETRIES
):

    for attempt in range(
        1,
        retries + 1
    ):

        try:

            data = yf.download(
                tickers=batch_symbols,
                period="1y",
                interval="1d",
                auto_adjust=False,
                progress=False,
                threads=True,
                group_by="ticker",
                timeout=30
            )

            if (
                data is not None
                and not data.empty
            ):

                return data

        except Exception as e:

            if attempt == retries:

                print(
                    f"批次下載失敗：{e}"
                )

        if attempt < retries:

            time.sleep(
                RETRY_DELAY
            )

    return None


# ============================================================
# 12. 批次下載 Yahoo Finance
# ============================================================

print()
print("【2/7】正在批次下載 Yahoo Finance 股價資料...")
print()


symbols = stocks[
    "symbol"
].tolist()


all_data = {}


download_start = time.time()


total_batches = (
    len(symbols) +
    BATCH_SIZE -
    1
) // BATCH_SIZE


for batch_number, start in enumerate(
    range(
        0,
        len(symbols),
        BATCH_SIZE
    ),
    start=1
):

    batch = symbols[
        start:
        start + BATCH_SIZE
    ]


    batch_data = download_batch(
        batch
    )


    if (
        batch_data is None
        or batch_data.empty
    ):

        print(
            f"批次 {batch_number}/"
            f"{total_batches}：沒有取得資料"
        )

        continue


    # ========================================================
    # 多股票資料
    # ========================================================

    if isinstance(
        batch_data.columns,
        pd.MultiIndex
    ):

        available_symbols = (
            batch_data
            .columns
            .get_level_values(0)
            .unique()
        )


        for symbol in batch:

            if symbol not in available_symbols:

                continue


            try:

                df = batch_data[
                    symbol
                ].copy()


                if (
                    df is not None
                    and not df.empty
                ):

                    all_data[
                        symbol
                    ] = df

            except Exception:

                continue


    # ========================================================
    # 單股票資料
    # ========================================================

    else:

        if len(batch) == 1:

            all_data[
                batch[0]
            ] = batch_data


    print(
        f"批次 {batch_number}/"
        f"{total_batches} 完成，"
        f"目前取得 {len(all_data):,} 檔"
    )


download_time = (
    time.time() -
    download_start
)


print()
print(
    f"成功取得資料："
    f"{len(all_data):,} 檔"
)

print(
    f"下載時間："
    f"{download_time:.1f} 秒"
)


# ============================================================
# 13. 第一層：流動性篩選
# ============================================================

print()
print("【3/7】正在進行第一層流動性篩選...")
print(
    f"條件：30日平均成交量 > "
    f"{MIN_AVG_VOLUME_SHARES / 1000:,.0f} 張"
)


liquidity_pass = {}


liquidity_start = time.time()


for symbol, df in all_data.items():

    try:

        if "Volume" not in df.columns:

            continue


        volume = (
            pd.to_numeric(
                df["Volume"],
                errors="coerce"
            )
            .dropna()
        )


        if len(volume) < 220:

            continue


        # 不包含今日
        avg_volume_30 = (
            volume
            .shift(1)
            .rolling(
                VOLUME_PERIOD
            )
            .mean()
            .iloc[-1]
        )


        if (
            pd.notna(avg_volume_30)
            and
            avg_volume_30 >
            MIN_AVG_VOLUME_SHARES
        ):

            liquidity_pass[
                symbol
            ] = df


    except Exception:

        continue


liquidity_time = (
    time.time() -
    liquidity_start
)


print()
print(
    f"30日平均成交量 > 2,000張："
    f"{len(liquidity_pass):,} 檔"
)

print(
    f"篩選時間："
    f"{liquidity_time:.1f} 秒"
)


# ============================================================
# 14. 計算技術指標
# ============================================================

print()
print("【4/7】正在計算 RSI / ADX / MA...")
print()


results = []


calc_start = time.time()


for symbol, df in tqdm(
    liquidity_pass.items(),
    desc="計算技術指標"
):

    try:

        required_columns = [
            "High",
            "Low",
            "Close",
            "Volume"
        ]


        if not all(
            column in df.columns
            for column in required_columns
        ):

            continue


        df = df[
            required_columns
        ].copy()


        # 數值化
        for column in required_columns:

            df[column] = pd.to_numeric(
                df[column],
                errors="coerce"
            )


        df = df.dropna()


        # 至少需要220筆資料
        if len(df) < 220:

            continue


        # ====================================================
        # MA20
        # ====================================================

        df["MA20"] = (
            df["Close"]
            .rolling(
                MA20_PERIOD
            )
            .mean()
        )


        # ====================================================
        # MA50
        # ====================================================

        df["MA50"] = (
            df["Close"]
            .rolling(
                MA50_PERIOD
            )
            .mean()
        )


        # ====================================================
        # MA200
        # ====================================================

        df["MA200"] = (
            df["Close"]
            .rolling(
                MA200_PERIOD
            )
            .mean()
        )


        # ====================================================
        # RSI14
        # ====================================================

        df["RSI14"] = calculate_rsi(
            df["Close"],
            RSI_PERIOD
        )


        # ====================================================
        # ADX14
        # ====================================================

        df["ADX14"] = calculate_adx(
            df["High"],
            df["Low"],
            df["Close"],
            ADX_PERIOD
        )


        # ====================================================
        # 30日平均成交量
        # 不包含今日
        # ====================================================

        df["VolumeMA30"] = (
            df["Volume"]
            .shift(1)
            .rolling(
                VOLUME_PERIOD
            )
            .mean()
        )


        # ====================================================
        # 最新資料
        # ====================================================

        latest = df.iloc[-1]


        close = float(
            latest["Close"]
        )

        ma20 = float(
            latest["MA20"]
        )

        ma50 = float(
            latest["MA50"]
        )

        ma200 = float(
            latest["MA200"]
        )

        rsi14 = float(
            latest["RSI14"]
        )

        adx14 = float(
            latest["ADX14"]
        )

        today_volume = float(
            latest["Volume"]
        )

        avg_volume_30 = float(
            latest["VolumeMA30"]
        )


        # ====================================================
        # 檢查指標
        # ====================================================

        values = [
            close,
            ma20,
            ma50,
            ma200,
            rsi14,
            adx14,
            today_volume,
            avg_volume_30
        ]


        if not all(
            np.isfinite(value)
            for value in values
        ):

            continue


        # ====================================================
        # 6個月績效
        # ====================================================

        if len(df) <= SIX_MONTH_DAYS:

            continue


        old_close = float(
            df["Close"]
            .iloc[
                -SIX_MONTH_DAYS - 1
            ]
        )


        if old_close <= 0:

            continue


        performance_6m = (
            (
                close /
                old_close
            ) - 1
        ) * 100


        # ====================================================
        # 六項條件
        # ====================================================

        condition_1 = (
            performance_6m <
            MAX_6M_RETURN
        )


        condition_2 = (
            rsi14 >
            RSI_MIN
        )


        condition_3 = (
            adx14 >
            ADX_MIN
        )


        condition_4 = (
            ma50 >
            ma200
        )


        condition_5 = (
            today_volume >
            avg_volume_30
            and
            avg_volume_30 >
            MIN_AVG_VOLUME_SHARES
        )


        condition_6 = (
            close <
            ma20 *
            MA20_FACTOR
        )


        # ====================================================
        # 條件通過數
        # ====================================================

        condition_list = [
            condition_1,
            condition_2,
            condition_3,
            condition_4,
            condition_5,
            condition_6
        ]


        pass_count = sum(
            condition_list
        )


        # ====================================================
        # 額外資訊
        # ====================================================

        ma20_distance = (
            (
                close /
                ma20
            ) - 1
        ) * 100


        volume_ratio = (
            today_volume /
            avg_volume_30
        )


        # ====================================================
        # 股票基本資料
        # ====================================================

        if symbol not in stock_info.index:

            continue


        info = stock_info.loc[
            symbol
        ]


        stock_id = info[
            "stock_id"
        ]


        stock_name = info[
            "stock_name"
        ]


        market = info[
            "type"
        ]


        # ====================================================
        # 建立完整結果
        # ====================================================

        results.append({

            "股票代號":
                stock_id,

            "股票名稱":
                stock_name,

            "市場":
                market,

            "日期":
                df.index[-1].strftime(
                    "%Y-%m-%d"
                ),

            "收盤價":
                round(
                    close,
                    2
                ),

            "6個月績效%":
                round(
                    performance_6m,
                    2
                ),

            "RSI14":
                round(
                    rsi14,
                    2
                ),

            "ADX14":
                round(
                    adx14,
                    2
                ),

            "MA20":
                round(
                    ma20,
                    2
                ),

            "MA50":
                round(
                    ma50,
                    2
                ),

            "MA200":
                round(
                    ma200,
                    2
                ),

            "股價距MA20%":
                round(
                    ma20_distance,
                    2
                ),

            "今日成交量(張)":
                round(
                    today_volume / 1000
                ),

            "30日平均量(張)":
                round(
                    avg_volume_30 / 1000
                ),

            "成交量倍數":
                round(
                    volume_ratio,
                    2
                ),

            "符合條件數":
                pass_count,

            "條件1_6月績效":
                "✓" if condition_1 else "✗",

            "條件2_RSI":
                "✓" if condition_2 else "✗",

            "條件3_ADX":
                "✓" if condition_3 else "✗",

            "條件4_MA50_MA200":
                "✓" if condition_4 else "✗",

            "條件5_成交量":
                "✓" if condition_5 else "✗",

            "條件6_股價_MA20":
                "✓" if condition_6 else "✗"
        })


    except Exception:

        continue


calc_time = (
    time.time() -
    calc_start
)


# ============================================================
# 15. 建立完整結果
# ============================================================

print()
print("【5/7】正在整理選股結果...")


all_result_df = pd.DataFrame(
    results
)


# ============================================================
# 16. 排序
# ============================================================

if not all_result_df.empty:

    all_result_df = all_result_df.sort_values(
        by=[
            "符合條件數",
            "成交量倍數",
            "ADX14"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )


    all_result_df = (
        all_result_df
        .reset_index(drop=True)
    )


# ============================================================
# 17. 取得「所有條件符合 (6條件)」與「1~5條件符合」
# ============================================================

if not all_result_df.empty:

    # 所有條件符合
    result_df = (
        all_result_df[
            all_result_df["符合條件數"] == 6
        ]
        .copy()
        .reset_index(drop=True)
    )

    # 1~5條件都符合，但第6條件不符合
    part_condition_df = (
        all_result_df[
            (all_result_df["符合條件數"] == 5) &
            (all_result_df["條件6_股價_MA20"] == "✗")
        ]
        .copy()
        .reset_index(drop=True)
    )

else:

    result_df = pd.DataFrame()
    part_condition_df = pd.DataFrame()


# ============================================================
# 20. 顯示結果
# ============================================================

print()
print("=" * 140)
print("★★★★★ 台股選股（六條件） ★★★★★")
print("=" * 140)


if result_df.empty:

    print()
    print(
        "目前沒有股票同時符合全部六個條件。"
    )

else:

    print()
    print(
        f"★★★★★ 所有條件符合："
        f"{len(result_df)} 檔 ★★★★★"
    )

    print()

    display(
        result_df
    )


print()
print(
    f"1~5條件符合："
    f"{len(part_condition_df):,} 檔"
)


# ============================================================
# 21. Excel 檔名（含執行時間）
# ============================================================

print()
print("【6/7】正在建立 Excel 報表...")

exec_time_str = datetime.now().strftime("%Y%m%d")

output_excel = f"台股選股（六條件）_{exec_time_str}.xlsx"


# ============================================================
# 22. 建立 Excel（僅寫入指定兩個分頁）
# ============================================================

with pd.ExcelWriter(
    output_excel,
    engine="openpyxl"
) as writer:

    # 所有條件符合
    result_df.to_excel(
        writer,
        index=False,
        sheet_name="所有條件符合"
    )

    # 1~5條件符合
    part_condition_df.to_excel(
        writer,
        index=False,
        sheet_name="1~5條件符合"
    )


# ============================================================
# 23. Excel 格式化
# ============================================================

print("正在格式化 Excel...")


wb = load_workbook(
    output_excel
)


# ------------------------------------------------------------
# 工作表格式
# ------------------------------------------------------------

for ws in wb.worksheets:

    # 凍結第一列
    ws.freeze_panes = "A2"


    # 自動篩選
    if ws.max_row > 1:

        ws.auto_filter.ref = (
            ws.dimensions
        )


    # 標題列
    for cell in ws[1]:

        cell.font = Font(
            bold=True
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center"
        )


    # 所有儲存格置中
    for row in ws.iter_rows():

        for cell in row:

            cell.alignment = Alignment(
                vertical="center"
            )


    # 自動欄寬
    for column_cells in ws.columns:

        max_length = 0

        column_letter = (
            get_column_letter(
                column_cells[0].column
            )
        )

        for cell in column_cells:

            try:

                cell_length = len(
                    str(cell.value)
                )

                if cell_length > max_length:

                    max_length = cell_length

            except Exception:

                pass


        ws.column_dimensions[
            column_letter
        ].width = min(
            max(max_length + 2, 10),
            30
        )


# ============================================================
# 24. 分頁格式強化與表格美化
# ============================================================

if "所有條件符合" in wb.sheetnames:

    ws = wb["所有條件符合"]

    headers = {
        cell.value: cell.column
        for cell in ws[1]
    }

    if "符合條件數" in headers:
        col = headers["符合條件數"]
        for row in range(2, ws.max_row + 1):
            ws.cell(row=row, column=col).font = Font(bold=True)

    # 建立 Excel Table
    if ws.max_row > 1:
        table_ref = f"A1:{get_column_letter(ws.max_column)}{ws.max_row}"
        tab = Table(displayName="AllPassResult", ref=table_ref)
        style = TableStyleInfo(
            name="TableStyleMedium2",
            showFirstColumn=False,
            showLastColumn=False,
            showRowStripes=True,
            showColumnStripes=False
        )
        tab.tableStyleInfo = style
        ws.add_table(tab)


if "1~5條件符合" in wb.sheetnames:

    ws = wb["1~5條件符合"]

    headers = {
        cell.value: cell.column
        for cell in ws[1]
    }

    if "符合條件數" in headers:
        col = headers["符合條件數"]
        for row in range(2, ws.max_row + 1):
            ws.cell(row=row, column=col).font = Font(bold=True)


# ============================================================
# 25. 儲存 Excel
# ============================================================

wb.save(
    output_excel
)


# ============================================================
# 26. 最終統計
# ============================================================

total_time = (
    time.time() -
    program_start
)


print()
print("=" * 140)
print("★★★★★ 台股選股（六條件）完成 ★★★★★")
print("=" * 140)


print()
print(
    f"股票總數："
    f"{len(stocks):,} 檔"
)


print(
    f"成功取得資料："
    f"{len(all_data):,} 檔"
)


print(
    f"平均成交量 > 2,000張："
    f"{len(liquidity_pass):,} 檔"
)


print(
    f"1~5條件符合："
    f"{len(part_condition_df):,} 檔"
)


print(
    f"所有條件符合："
    f"{len(result_df):,} 檔"
)


print()
print(
    f"資料下載時間："
    f"{download_time:.1f} 秒"
)


print(
    f"技術指標計算時間："
    f"{calc_time:.1f} 秒"
)


print(
    f"程式總執行時間："
    f"{total_time:.1f} 秒"
)


print()
print(
    f"Excel檔名："
    f"{output_excel}"
)


print("=" * 140)


# ============================================================
# 28. Colab 自動下載 Excel
# ============================================================

try:

    from google.colab import files

    files.download(
        output_excel
    )

except Exception:

    print(
        "目前環境不是 Google Colab，"
        "因此略過自動下載。"
    )


★★★★★ 台股選股（六條件） ★★★★★

選股條件：
① 6個月績效 < 50%
② RSI(14) > 50
③ ADX(14) > 20
④ MA50 > MA200
⑤ 今日成交量 > 30日平均量，且平均量 > 2,000張
⑥ 股價 < MA20 × 0.99

【1/7】正在取得台股股票清單...
上市 + 上櫃普通股票：2,153 檔

正在建立 Yahoo Finance 股票代號...

【2/7】正在批次下載 Yahoo Finance 股價資料...



ERROR:yfinance:
21 Failed downloads:
ERROR:yfinance:['6247.TWO', '6426.TWO', '3144.TWO', '1258.TWO', '3652.TWO', '4429.TWO', '3092.TWO', '8406.TWO', '4152.TWO', '5820.TWO', '5102.TWO', '3642.TWO', '2928.TWO', '5304.TWO', '8934.TWO', '4803.TWO', '6438.TWO', '6497.TWO', '5349.TWO', '6238.TWO', '6594.TWO']: YFPricesMissingError('possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")')


批次 1/22 完成，目前取得 100 檔


ERROR:yfinance:
8 Failed downloads:
ERROR:yfinance:['6404.TWO', '3089.TWO', '5281.TWO', '8420.TWO', '4944.TWO', '6472.TWO', '8476.TWO', '6806.TW']: YFPricesMissingError('possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")')


批次 2/22 完成，目前取得 200 檔


...

批次 5/22 完成，目前取得 500 檔
批次 6/22 完成，目前取得 600 檔
批次 7/22 完成，目前取得 700 檔
批次 8/22 完成，目前取得 800 檔
批次 9/22 完成，目前取得 900 檔
批次 10/22 完成，目前取得 1,000 檔
批次 11/22 完成，目前取得 1,100 檔
批次 12/22 完成，目前取得 1,200 檔
批次 13/22 完成，目前取得 1,300 檔
批次 14/22 完成，目前取得 1,400 檔
批次 15/22 完成，目前取得 1,500 檔
批次 16/22 完成，目前取得 1,600 檔
批次 17/22 完成，目前取得 1,700 檔
批次 18/22 完成，目前取得 1,800 檔
批次 19/22 完成，目前取得 1,900 檔
批次 20/22 完成，目前取得 2,000 檔
批次 21/22 完成，目前取得 2,100 檔
批次 22/22 完成，目前取得 2,153 檔

成功取得資料：2,153 檔
下載時間：124.1 秒

【3/7】正在進行第一層流動性篩選...
條件：30日平均成交量 > 2,000 張

30日平均成交量 > 2,000張：379 檔
篩選時間：1.0 秒

【4/7】正在計算 RSI / ADX / MA...



計算技術指標:   0%|          | 0/379 [00:00<?, ?it/s]


【5/7】正在整理選股結果...

★★★★★ 台股選股（六條件） ★★★★★

目前沒有股票同時符合全部六個條件。

1~5條件符合：24 檔

【6/7】正在建立 Excel 報表...
正在格式化 Excel...

★★★★★ 台股選股（六條件）完成 ★★★★★

股票總數：2,153 檔
成功取得資料：2,153 檔
平均成交量 > 2,000張：379 檔
1~5條件符合：24 檔
所有條件符合：0 檔

資料下載時間：124.1 秒
技術指標計算時間：5.8 秒
程式總執行時間：131.8 秒

Excel檔名：台股選股（六條件）_20261005.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### **均線法則**

In [ ]:
# @title
import datetime
import time
from google.colab import userdata
import pandas as pd
import pytz
import requests
import yfinance as yf

# ==========================================
# 1. 憑證設定 (優先讀取 Colab Secrets)
# ==========================================
try:
  LINE_CHANNEL_ACCESS_TOKEN = userdata.get('LINE_TOKEN')
  LINE_USER_ID = userdata.get('LINE_USER_ID')
except Exception:
  # 若未於 Secrets 設定 LINE_TOKEN 與 LINE_USER_ID，可在此貼上字串
  LINE_CHANNEL_ACCESS_TOKEN = "/JvwwTYrzi4Ao4twBsWBuo1jTNCpAHIWEVxLBVXfjduSdsKMbvnt+HBBZN2q/vuMobm4O2M0U1P8s8rdl8t8bBp6sl9bO4KMTY09VOas14v4O+ao8VNPuqquWekeL755BFG7fQ1y/bYlbnOH/TNbngdB04t89/1O/w1cDnyilFU="
  LINE_USER_ID = "U13239c411c6e3bd5c44ff791f8f35e37"

TW_TZ = pytz.timezone("Asia/Taipei")

# ==========================================
# 2. 39 檔監控股票清單與均線參數
# ==========================================
STOCKS = {
    "1504": {"name": "東元", "ma": 23, "type": ".TW"},
    "1711": {"name": "永光", "ma": 30, "type": ".TW"},
    "1795": {"name": "美時", "ma": 23, "type": ".TW"},
    "2301": {"name": "光寶科", "ma": 20, "type": ".TW"},
    "2303": {"name": "聯電", "ma": 27, "type": ".TW"},
    "2308": {"name": "台達電", "ma": 26, "type": ".TW"},
    "2313": {"name": "華通", "ma": 19, "type": ".TW"},
    "2317": {"name": "鴻海", "ma": 19, "type": ".TW"},
    "2324": {"name": "仁寶", "ma": 19, "type": ".TW"},
    "2327": {"name": "國巨", "ma": 19, "type": ".TW"},
    "2330": {"name": "台積電", "ma": 26, "type": ".TW"},
    "2337": {"name": "旺宏", "ma": 30, "type": ".TW"},
    "2338": {"name": "光罩", "ma": 26, "type": ".TW"},
    "2344": {"name": "華邦電", "ma": 19, "type": ".TW"},
    "2345": {"name": "智邦", "ma": 30, "type": ".TW"},
    "2347": {"name": "聯強", "ma": 38, "type": ".TW"},
    "2353": {"name": "宏碁", "ma": 17, "type": ".TW"},
    "2367": {"name": "耀華", "ma": 26, "type": ".TW"},
    "2376": {"name": "技嘉", "ma": 30, "type": ".TW"},
    "2382": {"name": "廣達", "ma": 23, "type": ".TW"},
    "2383": {"name": "台光電", "ma": 22, "type": ".TW"},
    "2385": {"name": "群光", "ma": 25, "type": ".TW"},
    "2395": {"name": "研華", "ma": 22, "type": ".TW"},
    "2408": {"name": "南亞科", "ma": 14, "type": ".TW"},
    "2409": {"name": "友達", "ma": 18, "type": ".TW"},
    "2454": {"name": "聯發科", "ma": 29, "type": ".TW"},
    "2472": {"name": "立隆電", "ma": 24, "type": ".TW"},
    "2485": {"name": "兆赫", "ma": 27, "type": ".TW"},
    "2498": {"name": "宏達電", "ma": 23, "type": ".TW"},
    "2610": {"name": "華航", "ma": 17, "type": ".TW"},
    "2646": {"name": "星宇航空", "ma": 23, "type": ".TW"},
    "3008": {"name": "大立光", "ma": 25, "type": ".TW"},
    "3017": {"name": "奇鋐", "ma": 19, "type": ".TW"},
    "3231": {"name": "緯創", "ma": 26, "type": ".TW"},
    "3450": {"name": "聯鈞", "ma": 20, "type": ".TW"},
    "3481": {"name": "群創", "ma": 20, "type": ".TW"},
    "3707": {"name": "漢磊", "ma": 18, "type": ".TW"},
    "3711": {"name": "日月光投控", "ma": 31, "type": ".TW"},
    "5347": {"name": "世界", "ma": 28, "type": ".TW"},
    "6116": {"name": "彩晶", "ma": 16, "type": ".TW"},
    "6191": {"name": "精成科", "ma": 18, "type": ".TW"},
    "6284": {"name": "佳邦", "ma": 24, "type": ".TW"},
    "6443": {"name": "元晶", "ma": 19, "type": ".TW"},
    "8069": {"name": "元太", "ma": 22, "type": ".TWO"},
}


# ==========================================
# 3. LINE 推播函式
# ==========================================
def send_line_push(msg_text):
  url = "https://api.line.me/v2/bot/message/push"
  headers = {
      "Content-Type": "application/json",
      "Authorization": f"Bearer {LINE_CHANNEL_ACCESS_TOKEN}",
  }
  payload = {
      "to": LINE_USER_ID,
      "messages": [{"type": "text", "text": msg_text}],
  }
  try:
    res = requests.post(url, headers=headers, json=payload, timeout=10)
    return res.status_code == 200
  except Exception as e:
    print(f"LINE 發送失敗: {e}")
    return False


# ==========================================
# 4. 盤中數據計算與條件觸發檢查
# ==========================================
def check_stock_alerts(alerted_today):
  now_tw = datetime.datetime.now(TW_TZ)
  print(
      f"[{now_tw.strftime('%Y-%m-%d %H:%M:%S')}] 開始掃描 39 檔台股價格..."
  )

  for code, info in STOCKS.items():
    if code in alerted_today:
      continue  # 當日已觸發提醒則跳過，防止重複洗版

    ticker_symbol = f"{code}{info['type']}"
    ma_period = info["ma"]
    stock_name = info["name"]

    try:
      stock = yf.Ticker(ticker_symbol)
      df = stock.history(period="60d", interval="1d")

      if len(df) < ma_period + 1:
        continue

      # 前一日均值 (計算不包含當日盤中即時日 K 的歷史數據)
      yesterday_df = df.iloc[:-1]
      prev_ma = yesterday_df["Close"].tail(ma_period).mean()
      threshold_price = round(prev_ma * 1.01, 2)  # 突破目標價

      current_price = round(df["Close"].iloc[-1], 2)
      current_vol = int(df["Volume"].iloc[-1] // 1000)
      yesterday_vol = int(df["Volume"].iloc[-2] // 1000)

      # 觸發條件：當前股價 >= 前一日均值 1.01 倍
      if current_price >= threshold_price:
        message = (
            f"📈 【台股均線突破提醒】\n"
            f"------------------------\n"
            f"1. 股票名稱 (代號)：{stock_name} ({code})\n"
            f"2. 當前股價：{current_price} 元\n"
            f"3. 突破價格 (前日{ma_period}MA x 1.01)：{threshold_price} 元\n"
            f"4. 成交量資訊：\n"
            f"   • 當前成交量：{current_vol:,} 張\n"
            f"   • 昨天成交量：{yesterday_vol:,} 張"
        )

        if send_line_push(message):
          print(f"✅ 成功發送推播：{stock_name} ({code})")
          alerted_today.add(code)

    except Exception as e:
      print(f"❌ 處理 {code} {stock_name} 時出錯: {e}")

    time.sleep(0.3)


# ==========================================
# 5. 主執行迴圈 (開盤時間監控)
# ==========================================
alerted_today = set()
last_reset_day = datetime.datetime.now(TW_TZ).date()

print("🚀 盤中均線監控系統已啟動...")

while True:
  now_tw = datetime.datetime.now(TW_TZ)

  # 每日跨日重置提醒清單
  if now_tw.date() != last_reset_day:
    alerted_today.clear()
    last_reset_day = now_tw.date()

  # 台股交易時間：週一至週五 09:00 - 13:30
  is_weekday = now_tw.weekday() < 5
  in_trading_hours = (
      (now_tw.hour == 9 and now_tw.minute >= 0)
      or (10 <= now_tw.hour <= 12)
      or (now_tw.hour == 13 and now_tw.minute <= 30)
  )

  if is_weekday and in_trading_hours:
    check_stock_alerts(alerted_today)
    time.sleep(300)  # 盤中每 5 分鐘掃描一次
  else:
    print(
        f"[{now_tw.strftime('%H:%M:%S')}] 非交易時間，等待中... (每 10"
        " 分鐘檢查一次)"
    )
    time.sleep(600)

🚀 盤中均線監控系統已啟動...
[22:56:21] 非交易時間，等待中... (每 10 分鐘檢查一次)


KeyboardInterrupt: 